In [ ]:
import phasespace
import numpy as np

import pandas as pd

import detector_simulation_tools as dst

In [ ]:
MASS_PARENT = 100 # GeV/c^2
MASS_CHILD = 10   # GeV/c^2

pmag = 500 # GeV/c

nevents_to_generate = 10

boost_vector = np.array([0,0, pmag, np.sqrt(pmag**2 + MASS_PARENT**2)])
boost_vectors = np.tile(boost_vector, (nevents_to_generate,1))

print("Created the boost vector and tiled it so that we have a boost vector for each decay we are simulating")
print()
print(boost_vectors)
print()

print(f"Generating {nevents_to_generate} decays")

weights, particles = phasespace.nbody_decay(MASS_PARENT, \
                                            [MASS_CHILD, MASS_CHILD]).generate(n_events=nevents_to_generate, boost_to=boost_vectors)

print("Generated the decays!")
print()
print(particles)

In [ ]:
import numpy as np

def ray_cylinder_intersection(
    origins: np.ndarray,      # Shape (N, 3) - starting points
    directions: np.ndarray,   # Shape (N, 3) - momentum/direction vectors (not necessarily normalized)
    radius: float,            # Cylinder radius
    half_length: float        # Half the cylinder length (extends from -half_length to +half_length on z-axis)
) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute entry and exit points for rays intersecting a cylinder centered at origin,
    aligned along the z-axis.
    
    Parameters
    ----------
    origins : ndarray of shape (N, 3)
        Starting points of the rays (outside the cylinder)
    directions : ndarray of shape (N, 3)
        Direction vectors (momentum vectors) of the rays
    radius : float
        Radius of the cylinder
    half_length : float
        Half-length of the cylinder (z ranges from -half_length to +half_length)
    
    Returns
    -------
    entry_points : ndarray of shape (N, 3)
        Entry points into the cylinder (NaN for rays that miss)
    exit_points : ndarray of shape (N, 3)
        Exit points from the cylinder (NaN for rays that miss)
    """
    origins = np.atleast_2d(origins)
    directions = np.atleast_2d(directions)
    
    N = origins.shape[0]
    
    # Initialize output arrays with NaN
    entry_points = np.full((N, 3), np.nan)
    exit_points = np.full((N, 3), np.nan)
    
    # Extract components
    ox, oy, oz = origins[:, 0], origins[:, 1], origins[:, 2]
    dx, dy, dz = directions[:, 0], directions[:, 1], directions[:, 2]
    
    # --- Curved surface intersection ---
    # Solve (ox + t*dx)² + (oy + t*dy)² = R²
    # This is a quadratic: a*t² + b*t + c = 0
    a = dx**2 + dy**2
    b = 2 * (ox * dx + oy * dy)
    c = ox**2 + oy**2 - radius**2
    
    discriminant = b**2 - 4 * a * c
    
    # For each ray, we'll collect valid t values
    # We need to handle curved surface and end caps separately
    
    t_candidates = np.full((N, 4), np.inf)  # Up to 4 candidate t values per ray
    
    # Curved surface intersections (where discriminant >= 0 and a != 0)
    valid_curved = (discriminant >= 0) & (np.abs(a) > 1e-12)
    sqrt_disc = np.sqrt(np.maximum(discriminant, 0))
    
    t1 = np.where(valid_curved, (-b - sqrt_disc) / (2 * a), np.inf)
    t2 = np.where(valid_curved, (-b + sqrt_disc) / (2 * a), np.inf)
    
    # Check if curved surface intersections are within z bounds
    z1 = oz + t1 * dz
    z2 = oz + t2 * dz
    
    t1_valid = valid_curved & (t1 >= 0) & (np.abs(z1) <= half_length)
    t2_valid = valid_curved & (t2 >= 0) & (np.abs(z2) <= half_length)
    
    t_candidates[:, 0] = np.where(t1_valid, t1, np.inf)
    t_candidates[:, 1] = np.where(t2_valid, t2, np.inf)
    
    # --- End cap intersections ---
    # Top cap: z = +half_length, solve oz + t*dz = half_length
    # Bottom cap: z = -half_length
    
    # Avoid division by zero
    dz_safe = np.where(np.abs(dz) > 1e-12, dz, np.inf)
    
    t_top = (half_length - oz) / dz_safe
    t_bottom = (-half_length - oz) / dz_safe
    
    # Check if cap intersections are within radius
    x_top = ox + t_top * dx
    y_top = oy + t_top * dy
    x_bottom = ox + t_bottom * dx
    y_bottom = oy + t_bottom * dy
    
    r2_top = x_top**2 + y_top**2
    r2_bottom = x_bottom**2 + y_bottom**2
    
    t_top_valid = (t_top >= 0) & (r2_top <= radius**2) & (np.abs(dz) > 1e-12)
    t_bottom_valid = (t_bottom >= 0) & (r2_bottom <= radius**2) & (np.abs(dz) > 1e-12)
    
    t_candidates[:, 2] = np.where(t_top_valid, t_top, np.inf)
    t_candidates[:, 3] = np.where(t_bottom_valid, t_bottom, np.inf)
    
    # --- Find the two smallest positive t values (entry and exit) ---
    t_sorted = np.sort(t_candidates, axis=1)
    
    t_entry = t_sorted[:, 0]
    t_exit = t_sorted[:, 1]
    
    # Rays that hit have finite entry and exit times
    valid_hit = np.isfinite(t_entry) & np.isfinite(t_exit)
    
    # Compute intersection points
    entry_points[valid_hit] = origins[valid_hit] + t_entry[valid_hit, np.newaxis] * directions[valid_hit]
    exit_points[valid_hit] = origins[valid_hit] + t_exit[valid_hit, np.newaxis] * directions[valid_hit]
    
    return entry_points, exit_points

In [ ]:
np.tan(np.deg2rad(91))

In [ ]:
# Claude generation
import numpy as np

def generate_upward_muon_momenta(
    n: int,
    p_magnitude: float,
    rng: np.random.Generator = None
) -> dict:
    """
    Generate muon momenta uniformly distributed in the upward (+y) hemisphere.
    
    In CMS coordinates:
    - z-axis: beam direction
    - x-axis: horizontal, toward LHC center
    - y-axis: vertical, pointing UP
    
    "Upward" means py > 0.
    
    Parameters
    ----------
    n : int
        Number of muons to generate
    p_magnitude : float
        Fixed magnitude of momentum for all muons
    rng : numpy.random.Generator, optional
        Random number generator (for reproducibility)
        
    Returns
    -------
    dict with keys:
        'px', 'py', 'pz': Cartesian momentum components
        'pt': transverse momentum
        'theta': polar angle (from +z axis, CMS convention)
        'phi': azimuthal angle (in x-y plane from +x axis)
        'eta': pseudorapidity
        'momentum_vectors': shape (n, 3) array of [px, py, pz]
    """
    if rng is None:
        rng = np.random.default_rng()
    
    # Method: Generate uniform points on full sphere, then constrain to py > 0
    # 
    # For uniform distribution on a sphere:
    #   - phi_spherical = uniform(0, 2*pi)  [azimuthal around z-axis in standard coords]
    #   - cos(theta_spherical) = uniform(-1, 1)  [this gives uniform solid angle]
    #
    # But we want py > 0 hemisphere, so we use a different approach:
    # Generate uniform on sphere, then rotate or use rejection/direct sampling.
    #
    # Cleanest method: sample uniformly on unit sphere, keep only py > 0
    # Or equivalently: sample the +y hemisphere directly.
    
    # Direct method for +y hemisphere:
    # We want py > 0, with uniform distribution over that hemisphere.
    #
    # Parameterize by angle from +y axis (call it alpha) and rotation around y-axis (call it beta):
    #   px = sin(alpha) * cos(beta)
    #   py = cos(alpha)              <- this is positive when alpha in [0, pi/2]
    #   pz = sin(alpha) * sin(beta)
    #
    # For uniform solid angle: sample cos(alpha) uniformly in [0, 1], beta uniformly in [0, 2*pi]
    
    cos_alpha = rng.uniform(0, 1, size=n)  # cos(alpha) in [0, 1] -> alpha in [0, pi/2]
    alpha = np.arccos(cos_alpha)           # angle from +y axis
    beta = rng.uniform(0, 2 * np.pi, size=n)  # rotation around y-axis
    
    # Unit direction vector
    px_unit = np.sin(alpha) * np.cos(beta)
    py_unit = cos_alpha  # = cos(alpha), always positive
    pz_unit = np.sin(alpha) * np.sin(beta)
    
    # Scale by momentum magnitude
    px = p_magnitude * px_unit
    py = p_magnitude * py_unit
    pz = p_magnitude * pz_unit
    
    # Convert to CMS coordinates
    #cms = cartesian_to_cms(px, py, pz)
    p,pt,eta,phi,theta = dst.cartesian_to_cms(px, py, pz)
    #p, pt, eta, phi, theta = 
    
    return {
        'px': px,
        'py': py,
        'pz': pz,
        'pt': pt,
        'theta': theta,
        'phi': phi,
        'eta': eta,
        'p': p,
        'momentum_vectors': np.column_stack([px, py, pz])
    }


def generate_upward_muon_momenta_cone(
    n: int,
    p_magnitude: float,
    max_angle_from_vertical: float = np.pi / 2,
    rng: np.random.Generator = None
) -> dict:
    """
    Generate muon momenta uniformly distributed in a cone around the +y (up) direction.
    
    This allows you to restrict to a narrower cone than the full hemisphere.
    
    Parameters
    ----------
    n : int
        Number of muons to generate
    p_magnitude : float
        Fixed magnitude of momentum for all muons
    max_angle_from_vertical : float
        Maximum angle from the +y axis in radians.
        pi/2 = full hemisphere (default)
        pi/4 = 45-degree cone
        pi/6 = 30-degree cone
    rng : numpy.random.Generator, optional
        Random number generator
        
    Returns
    -------
    dict with same keys as generate_upward_muon_momenta
    """
    if rng is None:
        rng = np.random.default_rng()
    
    # For uniform solid angle in a cone of half-angle alpha_max:
    # Sample cos(alpha) uniformly in [cos(alpha_max), 1]
    cos_alpha_min = np.cos(max_angle_from_vertical)
    
    cos_alpha = rng.uniform(cos_alpha_min, 1, size=n)
    alpha = np.arccos(cos_alpha)
    beta = rng.uniform(0, 2 * np.pi, size=n)
    
    # Unit direction vector (y is "up")
    px_unit = np.sin(alpha) * np.cos(beta)
    py_unit = cos_alpha
    pz_unit = np.sin(alpha) * np.sin(beta)
    
    # Scale by momentum magnitude
    px = p_magnitude * px_unit
    py = p_magnitude * py_unit
    pz = p_magnitude * pz_unit
    
    px,py,pz,p,pt,theta,eta = dst.cartesian_to_cms(px, py, pz)
    
    return {
        'px': px,
        'py': py,
        'pz': pz,
        'pt': pt,
        'theta': theta,
        'phi': phi,
        'eta': eta,
        'p': p,
        'momentum_vectors': np.column_stack([px, py, pz])
    }

In [ ]:
import matplotlib.pyplot as plt

# Generate muons
rng = np.random.default_rng(42)
muons = generate_upward_muon_momenta(10000, p_magnitude=100.0, rng=rng)

# Verify: all py should be positive
print(f"All py > 0: {np.all(muons['py'] > 0)}")
print(f"py range: [{muons['py'].min():.2f}, {muons['py'].max():.2f}]")
print(f"p magnitude check: {np.allclose(muons['p'], 100.0)}")

# Visualize the distribution
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 3D scatter of directions (unit vectors)
ax = fig.add_subplot(2, 2, 1, projection='3d')
sample = slice(0, 1000)  # Plot subset for clarity
ax.scatter(muons['px'][sample]/100, muons['py'][sample]/100, muons['pz'][sample]/100, 
           alpha=0.3, s=1)
ax.set_xlabel('px/p')
ax.set_ylabel('py/p')
ax.set_zlabel('pz/p')
ax.set_title('Direction vectors (should cover +y hemisphere)')

# Distribution of py (should be uniform in py for uniform solid angle!)
axes[0, 1].hist(muons['py'], bins=50, density=True)
axes[0, 1].set_xlabel('py')
axes[0, 1].set_ylabel('Density')
axes[0, 1].set_title('py distribution (should be uniform)')

# CMS phi distribution (should be uniform)
axes[1, 0].hist(muons['phi'], bins=50, density=True)
axes[1, 0].set_xlabel('phi (radians)')
axes[1, 0].set_title('CMS phi distribution')

# CMS eta distribution
axes[1, 1].hist(muons['eta'], bins=50, density=True)
axes[1, 1].set_xlabel('eta')
axes[1, 1].set_title('CMS eta distribution')

plt.tight_layout()
plt.show()

# Narrower cone example
muons_cone = generate_upward_muon_momenta_cone(
    5000, 
    p_magnitude=100.0, 
    max_angle_from_vertical=np.pi/6,  # 30-degree cone
    rng=rng
)
print(f"\n30-degree cone:")
print(f"  py range: [{muons_cone['py'].min():.2f}, {muons_cone['py'].max():.2f}]")
print(f"  Min py/p: {(muons_cone['py']/muons_cone['p']).min():.3f} (should be ~cos(30°)={np.cos(np.pi/6):.3f})")